# Pipeline de avaliação — resultados_ferramenta (teste seguro)

Este notebook é uma adaptação do pipeline antigo para a estrutura nova:

- notebook em `eval/notebooks/`
- perguntas em `eval/perguntas/`
- agente em `agent/`, na raiz do projeto
- saída em `eval/resultados_ferramenta/`

Por padrão, ele roda **somente 1 pergunta**, **somente no BNDES**, com **DeepSeek Flash**, e salva com um nome contendo `TESTE_DEEPSEEK_FLASH`, para não sobrescrever os `.xlsx` já existentes.

Depois que o teste passar, você pode:

1. trocar `MAX_PERGUNTAS = 1` por `MAX_PERGUNTAS = None`;
2. ativar os demais editais em `EDITAIS`;
3. remover ou mudar `RUN_TAG`, se quiser gerar os arquivos finais.


In [1]:
import os, sys, time, glob, re
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from dotenv import load_dotenv
from langchain_core.callbacks import BaseCallbackHandler


def encontrar_roots(start=None) -> tuple[Path, Path]:
    """
    Encontra:
      - PROJECT_ROOT: raiz do repositório, onde fica `agent/`
      - EVAL_ROOT: pasta da avaliação, onde ficam `perguntas/` e `resultados_ferramenta/`

    Cobre os dois layouts:

    Layout novo:
      edital-assistant/
        agent/
        eval/
          notebooks/
          perguntas/
          resultados_ferramenta/

    Layout antigo:
      evals/
        agent/
        notebooks/
        perguntas/
        resultados_ferramenta/
    """
    start = Path(start or Path.cwd()).resolve()

    tentativas: list[tuple[Path, Path]] = []
    for p in [start, *start.parents]:
        # Novo: p é a raiz do projeto e p/eval contém perguntas
        tentativas.append((p, p / "eval"))

        # Variante antiga: p é a raiz do projeto e p/evals contém perguntas
        tentativas.append((p, p / "evals"))

        # Antigo ou fallback: a própria pasta contém agent/ e perguntas/
        tentativas.append((p, p))

        # Caso o cwd esteja dentro de eval/ ou evals/: agent/ fica no pai, perguntas/ fica em p
        tentativas.append((p.parent, p))

    vistos = set()
    for project_root, eval_root in tentativas:
        project_root = project_root.resolve()
        eval_root = eval_root.resolve()
        chave = (project_root, eval_root)
        if chave in vistos:
            continue
        vistos.add(chave)

        if (project_root / "agent").is_dir() and (eval_root / "perguntas").is_dir():
            return project_root, eval_root

    raise RuntimeError(
        "Não encontrei a estrutura esperada. Ajuste manualmente, por exemplo:\n"
        "PROJECT_ROOT = Path('/home/julio/Documentos/tcc_GENAI/v8/edital-assistant')\n"
        "EVAL_ROOT = PROJECT_ROOT / 'eval'"
    )


PROJECT_ROOT, EVAL_ROOT = encontrar_roots()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from agent.agent import build_agent, ask

print(f"cwd:          {Path.cwd()}")
print(f"project_root: {PROJECT_ROOT}")
print(f"eval_root:    {EVAL_ROOT}")
print(f"env:          {PROJECT_ROOT / '.env'}")


cwd:          /home/julio/Documentos/tcc_GENAI/v8/edital-assistant
project_root: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant
eval_root:    /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval
env:          /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/.env


## TokenTracker — captura tokens + cache

In [2]:
class TokenTracker(BaseCallbackHandler):
    def __init__(self):
        self.reset()

    def reset(self):
        self.input_tokens = 0
        self.output_tokens = 0
        self.cache_read_tokens = 0
        self.cache_write_tokens = 0
        self.n_calls = 0

    def on_llm_end(self, response, **kwargs):
        self.n_calls += 1

        # 1) Formato OpenAI/DeepSeek via llm_output.token_usage
        usage = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
        if usage:
            self.input_tokens += usage.get("prompt_tokens", 0) or 0
            self.output_tokens += usage.get("completion_tokens", 0) or 0

            ds_hit = usage.get("prompt_cache_hit_tokens") or 0
            details = usage.get("prompt_tokens_details") or {}
            oai_hit = details.get("cached_tokens") or 0
            self.cache_read_tokens += max(ds_hit, oai_hit)
            return

        # 2) Formato LangChain novo via message.usage_metadata
        for gen_list in response.generations:
            for gen in gen_list:
                msg = getattr(gen, "message", None)
                um = getattr(msg, "usage_metadata", None) if msg else None
                if um:
                    self.input_tokens += um.get("input_tokens", 0) or 0
                    self.output_tokens += um.get("output_tokens", 0) or 0

                    details = um.get("input_token_details") or {}
                    self.cache_read_tokens += details.get("cache_read", 0) or 0
                    self.cache_write_tokens += details.get("cache_creation", 0) or 0
                    break

## Configuração do teste

A configuração abaixo é deliberadamente conservadora:

- `MODELOS`: apenas `deepseek-v4-flash`
- `EDITAIS`: apenas `bndes`
- `MAX_PERGUNTAS`: apenas 1 pergunta
- `RUN_TAG`: entra no nome do arquivo, evitando colisão com resultados reais
- `APAGAR_ANTIGOS`: `False`, para não remover nada já obtido

In [3]:
MODELOS = [
    ("deepseek", "deepseek-v4-flash"),
]

EDITAIS = [
    ("bndes", 1),
    # ("cvm", 2),
    # ("petrobras", 3),
]

# USD por 1M tokens. Ordem: (input_miss, input_cached, output).
PRECOS = {
    "openai/gpt-4o-mini":             (0.15,  0.075,  0.60),
    "openai/gpt-5.4-mini":            (0.25,  0.125,  2.00),
    "openai/gpt-5.4":                 (1.25,  0.625, 10.00),
    "openai/gpt-5.5":                 (5.00,  2.500, 30.00),
    "deepseek/deepseek-v4-flash":     (0.14,  0.028,  0.28),
    "deepseek/deepseek-v4-pro":       (1.74,  0.145,  3.48),
    "anthropic/claude-haiku-4-5":     (1.00,  1.000,  5.00),
    "anthropic/claude-sonnet-4-6":    (3.00,  3.000, 15.00),
    "anthropic/claude-opus-4-7":      (5.00,  5.000, 25.00),
}

PERGUNTAS_DIR = EVAL_ROOT / "perguntas"
RESULTADOS_DIR = EVAL_ROOT / "resultados_ferramenta"
RESULTADOS_DIR.mkdir(exist_ok=True)

RUN_TAG = "TESTE_DEEPSEEK_FLASH"
MAX_PERGUNTAS = 1              # mude para None para rodar todas
LIMITE_USD_POR_COMBO = 0.50    # teto baixo porque é só teste
APAGAR_ANTIGOS = False         # mantenha False para não apagar resultados reais

print(f"Perguntas:            {PERGUNTAS_DIR}")
print(f"Resultados:           {RESULTADOS_DIR}")
print(f"Modelos ativos:       {len(MODELOS)}")
print(f"Editais ativos:       {len(EDITAIS)}")
print(f"Total combinações:    {len(MODELOS) * len(EDITAIS)}")
print(f"Max perguntas/combo:  {MAX_PERGUNTAS}")
print(f"Tag de saída:         {RUN_TAG}")

Perguntas:            /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/perguntas
Resultados:           /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/resultados_ferramenta
Modelos ativos:       1
Editais ativos:       1
Total combinações:    1
Max perguntas/combo:  1
Tag de saída:         TESTE_DEEPSEEK_FLASH


## Validações gratuitas

In [4]:
# 1) Todos os modelos têm preço configurado
faltando_precos = [f"{p}/{m}" for p, m in MODELOS if f"{p}/{m}" not in PRECOS]
if faltando_precos:
    raise RuntimeError(f"Modelos sem preço em PRECOS: {faltando_precos}")

# 2) Arquivos de perguntas existem
faltando_perguntas = [
    str(PERGUNTAS_DIR / f"{edital_nome}.xlsx")
    for edital_nome, _ in EDITAIS
    if not (PERGUNTAS_DIR / f"{edital_nome}.xlsx").exists()
]
if faltando_perguntas:
    raise RuntimeError("Arquivos de perguntas não encontrados:\n- " + "\n- ".join(faltando_perguntas))

# 3) Conferência visual de arquivos já existentes: nada será apagado com APAGAR_ANTIGOS=False
print("OK — preços e arquivos de perguntas encontrados.\n")
print("Arquivos já existentes que parecem ser resultados finais:")
for edital_nome, _ in EDITAIS:
    encontrados = sorted(RESULTADOS_DIR.glob(f"{edital_nome}_*.xlsx"))
    print(f"\n{edital_nome}: {len(encontrados)} arquivo(s)")
    for arq in encontrados[:20]:
        print(f"  - {arq.name}")
    if len(encontrados) > 20:
        print(f"  ... +{len(encontrados) - 20} arquivo(s)")

OK — preços e arquivos de perguntas encontrados.

Arquivos já existentes que parecem ser resultados finais:

bndes: 9 arquivo(s)
  - bndes_claude-haiku-4-5.xlsx
  - bndes_claude-opus-4-7.xlsx
  - bndes_claude-sonnet-4-6.xlsx
  - bndes_deepseek-v4-flash.xlsx
  - bndes_deepseek-v4-pro.xlsx
  - bndes_gpt-4o-mini.xlsx
  - bndes_gpt-5.4-mini.xlsx
  - bndes_gpt-5.4.xlsx
  - bndes_gpt-5.5.xlsx


## Helpers

In [5]:
def slug(modelo: str) -> str:
    return modelo.replace("/", "_").replace(":", "_")


def custo_estimado(in_tok: int, out_tok: int, p_in: float, p_out: float) -> float:
    """Teto pessimista: trata todo input como cache miss."""
    return in_tok / 1_000_000 * p_in + out_tok / 1_000_000 * p_out


def custo_real(in_tok: int, out_tok: int, cache_read: int,
               p_in: float, p_in_cached: float, p_out: float) -> float:
    miss = max(in_tok - cache_read, 0)
    return (
        miss / 1_000_000 * p_in
        + cache_read / 1_000_000 * p_in_cached
        + out_tok / 1_000_000 * p_out
    )


def nome_saida(provider: str, modelo: str, edital_nome: str, sufixo: str = "") -> str:
    """
    Usa padrão parecido com resultados_ferramenta, mas com RUN_TAG e timestamp.
    Exemplo:
      bndes_deepseek-v4-flash_TESTE_DEEPSEEK_FLASH_20260101_120000.xlsx
    """
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    partes = [edital_nome, slug(modelo), RUN_TAG, ts]
    return "_".join(partes) + f"{sufixo}.xlsx"


def salvar_xlsx(provider: str, modelo: str, edital_nome: str, resultados: list[dict], sufixo: str = "") -> Path:
    df_r = pd.DataFrame(resultados)
    arq = RESULTADOS_DIR / nome_saida(provider, modelo, edital_nome, sufixo=sufixo)

    if arq.exists():
        raise FileExistsError(f"Arquivo já existe, abortando para não sobrescrever: {arq}")

    df_r.to_excel(arq, index=False)
    return arq


def apagar_antigos(provider: str, modelo: str, edital_nome: str) -> int:
    """
    Mantido para compatibilidade, mas só será chamado se APAGAR_ANTIGOS=True.
    Por padrão, não apaga nada.
    """
    padrao = str(RESULTADOS_DIR / f"{edital_nome}_{slug(modelo)}*.xlsx")
    n = 0
    for antigo in glob.glob(padrao):
        Path(antigo).unlink()
        n += 1
    return n

In [6]:
def carregar_perguntas(edital_nome: str) -> pd.DataFrame:
    arq_perguntas = PERGUNTAS_DIR / f"{edital_nome}.xlsx"
    df_q = pd.read_excel(arq_perguntas)

    if "pergunta" not in df_q.columns:
        raise RuntimeError(f"{arq_perguntas} não tem coluna 'pergunta'.")

    df_q = df_q.dropna(subset=["pergunta"])
    df_q = df_q[df_q["pergunta"].astype(str).str.strip() != ""].reset_index(drop=True)

    if MAX_PERGUNTAS is not None:
        df_q = df_q.head(MAX_PERGUNTAS).copy()

    if "id" not in df_q.columns:
        df_q.insert(0, "id", range(1, len(df_q) + 1))

    return df_q


def rodar_combo(provider: str, modelo: str, edital_nome: str, edital_id: int,
                limite_usd: float = LIMITE_USD_POR_COMBO):
    """
    Roda as perguntas configuradas. Retorna (xlsx_path, erro_fatal_ou_None).

    Erros em perguntas individuais vão para a coluna `erro` e a combinação continua.
    Estouro de orçamento ou erro fora do loop interrompem e salvam parcial.
    """
    chave = f"{provider}/{modelo}"
    p_in, p_in_cached, p_out = PRECOS[chave]

    df_q = carregar_perguntas(edital_nome)
    print(f"Perguntas carregadas para {edital_nome}: {len(df_q)}")

    agente = build_agent(provider=provider, model=modelo)
    tracker = TokenTracker()
    agente.llm = agente.llm.with_config(callbacks=[tracker])
    agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

    resultados = []
    custo_estim_acum = 0.0
    erro_fatal = None

    for i, row in df_q.iterrows():
        pergunta = row["pergunta"]
        t0 = time.time()
        erro, resposta = None, None

        tracker.reset()
        try:
            resposta = ask(
                agent=agente,
                question=pergunta,
                chat_history=[],
                edital_id_ativo=edital_id,
            )
        except Exception as e:
            erro = str(e)

        in_tok = tracker.input_tokens
        out_tok = tracker.output_tokens
        cache_read = tracker.cache_read_tokens
        n_calls = tracker.n_calls
        latencia = round(time.time() - t0, 2)

        c_estim = custo_estimado(in_tok, out_tok, p_in, p_out)
        c_real = custo_real(in_tok, out_tok, cache_read, p_in, p_in_cached, p_out)
        custo_estim_acum += c_estim

        resultados.append({
            "id": row["id"],
            "categoria": row.get("categoria", ""),
            "pergunta": pergunta,
            "resposta": resposta,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "total_tokens": in_tok + out_tok,
            "cache_read_tokens": cache_read,
            "cache_miss_tokens": max(in_tok - cache_read, 0),
            "n_invocacoes": n_calls,
            "custo_usd": round(c_estim, 6),
            "custo_real_usd": round(c_real, 6),
            "latencia_s": latencia,
            "erro": erro,
        })

        flag = "ERRO" if erro else "ok"
        cache_pct = (cache_read / in_tok * 100) if in_tok else 0
        print(
            f"      [{i+1:>2}/{len(df_q)}] {flag:>4}  {latencia:>5}s  "
            f"calls={n_calls}  in={in_tok:>5} (cache {cache_pct:>4.0f}%)  "
            f"out={out_tok:>4}  acum=${custo_estim_acum:.4f}"
        )

        if custo_estim_acum > limite_usd:
            erro_fatal = f"Estouro na pergunta {i+1}: ${custo_estim_acum:.4f} > ${limite_usd}"
            break

    sufixo = "_PARCIAL" if erro_fatal else ""
    arq = salvar_xlsx(provider, modelo, edital_nome, resultados, sufixo=sufixo) if resultados else None
    return arq, erro_fatal

## Execução do teste

Esta célula chama a API. Como `MAX_PERGUNTAS = 1`, o teste fica limitado a uma pergunta por combinação ativa.

In [7]:
sucessos, parciais, falhas = [], [], []

for provider, modelo in MODELOS:
    for edital_nome, edital_id in EDITAIS:
        if APAGAR_ANTIGOS:
            n = apagar_antigos(provider, modelo, edital_nome)
            print(f"\n[del]  {n} xlsx antigo(s) de {provider}/{modelo} × {edital_nome}")
        else:
            print("\n[seguro] APAGAR_ANTIGOS=False — nenhum arquivo antigo será removido.")

        print(f"[run]  {provider}/{modelo} × {edital_nome} (id={edital_id})")

        try:
            arq, erro_fatal = rodar_combo(provider, modelo, edital_nome, edital_id)
        except Exception as e:
            falhas.append((provider, modelo, edital_nome, str(e)))
            print(f"       ✗ FALHOU antes do xlsx: {e}")
            continue

        if erro_fatal:
            parciais.append((provider, modelo, edital_nome, arq, erro_fatal))
            print(f"       ⚠ PARCIAL — {erro_fatal}\n         salvo em {arq.name}")
        else:
            sucessos.append((provider, modelo, edital_nome, arq))
            print(f"       ✓ ok — salvo em {arq.name}")

print("\n" + "=" * 60)
print("=== SUMÁRIO DO TESTE ===")
print("=" * 60)
print(f"\n✓ Sucessos: {len(sucessos)}")
for p, m, e, a in sucessos:
    print(f"    {p}/{m} × {e}  →  {a.name}")

print(f"\n⚠ Parciais: {len(parciais)}")
for p, m, e, a, err in parciais:
    print(f"    {p}/{m} × {e}  →  {a.name}\n        {err}")

print(f"\n✗ Falhas: {len(falhas)}")
for p, m, e, err in falhas:
    print(f"    {p}/{m} × {e}\n        {err}")


[seguro] APAGAR_ANTIGOS=False — nenhum arquivo antigo será removido.
[run]  deepseek/deepseek-v4-flash × bndes (id=1)
Perguntas carregadas para bndes: 1
Carregando BGE-M3 (primeira vez pode demorar)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 51920.88it/s]


      [ 1/1]   ok  11.69s  calls=2  in= 8777 (cache   25%)  out= 251  acum=$0.0013
       ✓ ok — salvo em bndes_deepseek-v4-flash_TESTE_DEEPSEEK_FLASH_20260512_124228.xlsx

=== SUMÁRIO DO TESTE ===

✓ Sucessos: 1
    deepseek/deepseek-v4-flash × bndes  →  bndes_deepseek-v4-flash_TESTE_DEEPSEEK_FLASH_20260512_124228.xlsx

⚠ Parciais: 0

✗ Falhas: 0


## Conferir arquivos de teste gerados

In [8]:
arquivos_teste = sorted(RESULTADOS_DIR.glob(f"*{RUN_TAG}*.xlsx"))
print(f"{len(arquivos_teste)} arquivo(s) de teste encontrado(s) em {RESULTADOS_DIR}:\n")
for arq in arquivos_teste:
    print(f"- {arq.name}")

1 arquivo(s) de teste encontrado(s) em /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/resultados_ferramenta:

- bndes_deepseek-v4-flash_TESTE_DEEPSEEK_FLASH_20260512_124228.xlsx


## Sumário consolidado dos arquivos de teste

In [ ]:
linhas = []

for arq in sorted(RESULTADOS_DIR.glob(f"*{RUN_TAG}*.xlsx")):
    df = pd.read_excel(arq)

    if "cache_read_tokens" not in df.columns:
        df["cache_read_tokens"] = 0
    if "custo_real_usd" not in df.columns:
        df["custo_real_usd"] = df.get("custo_usd", 0)

    linhas.append({
        "arquivo": arq.name,
        "n_perguntas": len(df),
        "n_erros": int(df["erro"].notna().sum()) if "erro" in df.columns else 0,
        "input_tokens": int(df["input_tokens"].sum()) if "input_tokens" in df.columns else 0,
        "output_tokens": int(df["output_tokens"].sum()) if "output_tokens" in df.columns else 0,
        "cache_read": int(df["cache_read_tokens"].sum()),
        "custo_estim_usd": round(float(df["custo_usd"].sum()), 6) if "custo_usd" in df.columns else 0,
        "custo_real_usd": round(float(df["custo_real_usd"].sum()), 6),
        "latencia_med_s": round(float(df["latencia_s"].mean()), 2) if "latencia_s" in df.columns else None,
    })

df_sumario_teste = pd.DataFrame(linhas)
df_sumario_teste